# PyPSA-Eur

This notebook runs in the `pypsa-eur` environment. It checks the pinned
checkout, the pinned package set, and plans the small electricity test
workflow without downloading data.


In [ ]:
from pathlib import Path
import subprocess

import pypsa

REPO = Path("/work/models/pypsa-eur")
print("checkout :", (REPO / ".hem-ref").read_text().strip())
print("pypsa    :", pypsa.__version__)
print("config   :", (REPO / "config/test/config.electricity.yaml").exists())


## What the test workflow does

`config/test/config.electricity.yaml` is the upstream test case: Belgium only,
one week of 2013, with a reduced technology set. It downloads weather and
demand data the first time; the test case is much smaller than the full
model, but it is still not a trivial download. A larger run
(`config/config.default.yaml`) needs far more — use `/data` for that.


In [ ]:
result = subprocess.run(
    ["snakemake", "-n", "-c1", "--configfile", "config/test/config.electricity.yaml"],
    cwd=REPO,
    capture_output=True,
    text=True,
    timeout=600,
)
print(result.stdout[-4000:])
print(result.stderr[-2000:])
assert result.returncode == 0, "the workflow could not be planned"


## Run it for real

The solve itself is a Snakemake job. From a terminal in JupyterLab:

```bash
run-pypsa-eur.sh              # -c1, test config
CORES=all run-pypsa-eur.sh    # use every core
DRY_RUN=1 run-pypsa-eur.sh    # plan only
```

Results land in `/work/models/pypsa-eur/results/<run name>/`. The solved
network is a PyPSA `.nc` file, so you can open it with the cell below once a
run has finished.


In [ ]:
candidates = sorted(REPO.glob("results/**/*.nc"))
print("solved networks:", len(candidates))
for path in candidates[:10]:
    print(" ", path.relative_to(REPO))

# n = pypsa.Network(candidates[0])
# n.statistics.supply().head()
